In [ ]:
"""
================================================================================
SCRIPT OFICIAL DE GENERACIÓN Y ENRIQUECIMIENTO DE DATOS SINTÉTICOS
PROYECTO: Mantenimiento Predictivo Industrial (PredictiveMaintenance)
AUTOR: Dutaya (Senior Data Engineer - Predictive Maintenance & Industrial IoT)
SEMILLA GLOBAL (SEED): 42
================================================================================
Este script toma el archivo base de telemetría de 25 máquinas, elimina la fuga
de datos (leakage), enriquece con variables físicas de energía y odómetros,
construye los targets predictivos a 48 horas e inyecta imperfecciones realistas
(nulos, picos de ruido, caídas de red, drift y flatlines) con fines de evaluación.
"""

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# 0. CONFIGURACIÓN DE REPRODUCIBILIDAD Y RUTAS
# ------------------------------------------------------------------------------
SEMILLA = 42
np.random.seed(SEMILLA)

input_path = r"C:\Users\PC\Desktop\dataset_mantenimiento_predictivo.csv"
output_dir = r"C:\Users\PC\Desktop\mantenimiento_predictivo_realista\opcion_a_tabla_unica"
output_path = os.path.join(output_dir, "dataset_mantenimiento_predictivo_realista.csv")

os.makedirs(output_dir, exist_ok=True)

print("=" * 80)
print(f"INICIANDO PIPELINE DE GENERACIÓN CON REPRODUCIBILIDAD (SEED = {SEMILLA})")
print("=" * 80)

# Carga de datos base
print(f"[1/8] Cargando dataset base desde:\n      -> {input_path}")
df_raw = pd.read_csv(input_path)
df_raw["fecha_hora"] = pd.to_datetime(df_raw["fecha_hora"])

# ------------------------------------------------------------------------------
# 1. CATÁLOGO MAESTRO DE ACTIVOS (25 MÁQUINAS INDUSTRIALES)
# ------------------------------------------------------------------------------
print("[2/8] Incorporando catálogo maestro de activos y especificaciones de planta...")
catalogo_equipos = [
    {"id_maquina": "M-01", "tipo_equipo": "Torno CNC", "modelo": "Haas ST-30", "linea_produccion": "Linea_A_Mecanizado_Pesado", "antiguedad_anos": 7, "criticidad": "Alta", "costo_parada_hora_usd": 1500, "potencia_nominal_kw": 22.5, "marca": "Haas Automation"},
    {"id_maquina": "M-02", "tipo_equipo": "Torno CNC", "modelo": "Haas ST-30", "linea_produccion": "Linea_A_Mecanizado_Pesado", "antiguedad_anos": 5, "criticidad": "Alta", "costo_parada_hora_usd": 1500, "potencia_nominal_kw": 22.5, "marca": "Haas Automation"},
    {"id_maquina": "M-03", "tipo_equipo": "Torno CNC", "modelo": "Mazak QuickTurn", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 8, "criticidad": "Media", "costo_parada_hora_usd": 850, "potencia_nominal_kw": 18.0, "marca": "Mazak"},
    {"id_maquina": "M-04", "tipo_equipo": "Centro de Mecanizado 5 Ejes", "modelo": "DMG DMU 50", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 4, "criticidad": "Alta", "costo_parada_hora_usd": 2200, "potencia_nominal_kw": 30.0, "marca": "DMG Mori"},
    {"id_maquina": "M-05", "tipo_equipo": "Centro de Mecanizado 5 Ejes", "modelo": "DMG DMU 75", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 9, "criticidad": "Alta", "costo_parada_hora_usd": 2200, "potencia_nominal_kw": 30.0, "marca": "DMG Mori"},
    {"id_maquina": "M-06", "tipo_equipo": "Centro de Mecanizado 5 Ejes", "modelo": "Okuma Genos M560", "linea_produccion": "Linea_A_Mecanizado_Pesado", "antiguedad_anos": 6, "criticidad": "Alta", "costo_parada_hora_usd": 2200, "potencia_nominal_kw": 35.0, "marca": "Okuma"},
    {"id_maquina": "M-07", "tipo_equipo": "Centro de Mecanizado 5 Ejes", "modelo": "Okuma Genos M460", "linea_produccion": "Linea_A_Mecanizado_Pesado", "antiguedad_anos": 11, "criticidad": "Media", "costo_parada_hora_usd": 1200, "potencia_nominal_kw": 28.0, "marca": "Okuma"},
    {"id_maquina": "M-08", "tipo_equipo": "Fresadora Industrial", "modelo": "Bridgeport V1000", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 10, "criticidad": "Media", "costo_parada_hora_usd": 900, "potencia_nominal_kw": 15.0, "marca": "Bridgeport"},
    {"id_maquina": "M-09", "tipo_equipo": "Fresadora Industrial", "modelo": "Bridgeport V1000", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 3, "criticidad": "Baja", "costo_parada_hora_usd": 450, "potencia_nominal_kw": 15.0, "marca": "Bridgeport"},
    {"id_maquina": "M-10", "tipo_equipo": "Fresadora Industrial", "modelo": "Lagun FU-140", "linea_produccion": "Linea_A_Mecanizado_Pesado", "antiguedad_anos": 7, "criticidad": "Media", "costo_parada_hora_usd": 900, "potencia_nominal_kw": 18.5, "marca": "Lagun"},
    {"id_maquina": "M-11", "tipo_equipo": "Rectificadora Cilindrica", "modelo": "Studer S33", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 8, "criticidad": "Media", "costo_parada_hora_usd": 800, "potencia_nominal_kw": 11.0, "marca": "Studer"},
    {"id_maquina": "M-12", "tipo_equipo": "Rectificadora Cilindrica", "modelo": "Studer S21", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 12, "criticidad": "Baja", "costo_parada_hora_usd": 400, "potencia_nominal_kw": 9.5, "marca": "Studer"},
    {"id_maquina": "M-13", "tipo_equipo": "Rectificadora Plana", "modelo": "Chevalier Smart-H1224", "linea_produccion": "Linea_B_Mecanizado_Precision", "antiguedad_anos": 6, "criticidad": "Baja", "costo_parada_hora_usd": 400, "potencia_nominal_kw": 8.0, "marca": "Chevalier"},
    {"id_maquina": "M-14", "tipo_equipo": "Taladro Industrial Columna", "modelo": "Erlo TCA-50", "linea_produccion": "Linea_C_Corte_Y_Perforado", "antiguedad_anos": 9, "criticidad": "Baja", "costo_parada_hora_usd": 350, "potencia_nominal_kw": 5.5, "marca": "Erlo"},
    {"id_maquina": "M-15", "tipo_equipo": "Taladro Industrial Columna", "modelo": "Erlo TCA-35", "linea_produccion": "Linea_C_Corte_Y_Perforado", "antiguedad_anos": 5, "criticidad": "Baja", "costo_parada_hora_usd": 350, "potencia_nominal_kw": 5.5, "marca": "Erlo"},
    {"id_maquina": "M-16", "tipo_equipo": "Taladro Radial Pesado", "modelo": "Foradia GR-50", "linea_produccion": "Linea_C_Corte_Y_Perforado", "antiguedad_anos": 11, "criticidad": "Media", "costo_parada_hora_usd": 700, "potencia_nominal_kw": 7.5, "marca": "Foradia"},
    {"id_maquina": "M-17", "tipo_equipo": "Equipo Corte Laser Fibra", "modelo": "Bystronic ByStar 6kW", "linea_produccion": "Linea_C_Corte_Y_Perforado", "antiguedad_anos": 2, "criticidad": "Alta", "costo_parada_hora_usd": 2500, "potencia_nominal_kw": 40.0, "marca": "Bystronic"},
    {"id_maquina": "M-18", "tipo_equipo": "Equipo Corte Plasma HD", "modelo": "Hypertherm XPR300", "linea_produccion": "Linea_C_Corte_Y_Perforado", "antiguedad_anos": 6, "criticidad": "Media", "costo_parada_hora_usd": 1100, "potencia_nominal_kw": 25.0, "marca": "Hypertherm"},
    {"id_maquina": "M-19", "tipo_equipo": "Sierra Cinta Industrial", "modelo": "Kasto SSB 260", "linea_produccion": "Linea_C_Corte_Y_Perforado", "antiguedad_anos": 4, "criticidad": "Baja", "costo_parada_hora_usd": 300, "potencia_nominal_kw": 4.0, "marca": "Kasto"},
    {"id_maquina": "M-20", "tipo_equipo": "Compresor de Tornillo Principal", "modelo": "Atlas Copco GA 75 VSD", "linea_produccion": "Planta_Servicios_Auxiliares", "antiguedad_anos": 5, "criticidad": "Alta", "costo_parada_hora_usd": 3000, "potencia_nominal_kw": 75.0, "marca": "Atlas Copco"},
    {"id_maquina": "M-21", "tipo_equipo": "Compresor de Tornillo Auxiliar", "modelo": "Kaeser CSD 85", "linea_produccion": "Planta_Servicios_Auxiliares", "antiguedad_anos": 8, "criticidad": "Media", "costo_parada_hora_usd": 1400, "potencia_nominal_kw": 45.0, "marca": "Kaeser"},
    {"id_maquina": "M-22", "tipo_equipo": "Compresor de Respaldo", "modelo": "Ingersoll Rand R37", "linea_produccion": "Planta_Servicios_Auxiliares", "antiguedad_anos": 12, "criticidad": "Baja", "costo_parada_hora_usd": 500, "potencia_nominal_kw": 37.0, "marca": "Ingersoll Rand"},
    {"id_maquina": "M-23", "tipo_equipo": "Sistema Hidraulico Central A", "modelo": "Rexroth CytroBox", "linea_produccion": "Planta_Servicios_Auxiliares", "antiguedad_anos": 7, "criticidad": "Alta", "costo_parada_hora_usd": 2800, "potencia_nominal_kw": 55.0, "marca": "Bosch Rexroth"},
    {"id_maquina": "M-24", "tipo_equipo": "Sistema Hidraulico Central B", "modelo": "Parker EcoAir Pro", "linea_produccion": "Planta_Servicios_Auxiliares", "antiguedad_anos": 9, "criticidad": "Media", "costo_parada_hora_usd": 1300, "potencia_nominal_kw": 45.0, "marca": "Parker Hannifin"},
    {"id_maquina": "M-25", "tipo_equipo": "Unidad Hidraulica Auxiliar", "modelo": "Vickers PowerSys", "linea_produccion": "Planta_Servicios_Auxiliares", "antiguedad_anos": 10, "criticidad": "Baja", "costo_parada_hora_usd": 450, "potencia_nominal_kw": 18.5, "marca": "Vickers"}
]
df_cat = pd.DataFrame(catalogo_equipos)

# Eliminamos columnas sintéticas estáticas/precocinadas que inducen a leakage
cols_eliminar = [
    "vibracion_media_24h", "vibracion_std_24h",
    "temperatura_media_24h", "temperatura_std_24h", "presion_media_24h",
    "modelo", "antiguedad_anos", "criticidad", "costo_parada_hora_usd"
]
df = df_raw.drop(columns=[c for c in cols_eliminar if c in df_raw.columns])
df = df.merge(df_cat, on="id_maquina", how="left")

# ------------------------------------------------------------------------------
# 2. ENRIQUECIMIENTO: ENERGÍA, CORRIENTE Y ODÓMETROS OPERATIVOS
# ------------------------------------------------------------------------------
print("[3/8] Generando variables físicas de energía y odómetros...")

# Ordenamiento temporal estricto
df = df.sort_values(["id_maquina", "fecha_hora"]).reset_index(drop=True)

# A. Potencia Activa Consumida (kW)
# Sube cuando el estado_operativo == 1 y sufre sobreesfuerzo por falla inminente (+15%)
potencia_base = df["potencia_nominal_kw"] * (0.12 + 0.88 * (df["carga_pct"] / 100.0))
sobreconsumo = np.where(df["falla_proximas_48h"] == 1, 1.15, 1.0)
df["potencia_consumida_kw"] = np.where(df["estado_operativo"] == 1, potencia_base * sobreconsumo, 0.0)

# B. Corriente Eléctrica (Amperios trifásicos: I = P * 1000 / (sqrt(3) * V * cos_phi))
cos_phi = 0.88
df["corriente_a"] = np.where(
    df["estado_operativo"] == 1,
    (df["potencia_consumida_kw"] * 1000.0) / (np.sqrt(3) * df["voltaje_v"] * cos_phi),
    0.0
)

# C. Horas de operación totales acumuladas (Odómetro de vida útil)
base_horas = df["antiguedad_anos"] * 3650
horas_acum_campana = df.groupby("id_maquina")["estado_operativo"].cumsum()
df["horas_operacion_totales"] = base_horas + horas_acum_campana

# D. Ciclos de mecanizado acumulados
ciclos_por_hora = np.where(df["estado_operativo"] == 1, np.random.randint(3, 7, size=len(df)), 0)
df["_ciclos_tmp"] = ciclos_por_hora
df["ciclos_acumulados"] = (df["antiguedad_anos"] * 18000) + df.groupby("id_maquina")["_ciclos_tmp"].cumsum()
df.drop(columns=["_ciclos_tmp"], inplace=True)

# ------------------------------------------------------------------------------
# 3. REFACTORIZACIÓN SEMÁNTICA: DISPARO (261) VS CONVALECENCIA (1.661)
# ------------------------------------------------------------------------------
print("[4/8] Aplicando refactorización semántica de averías industriales...")
df.rename(columns={
    "evento_falla": "falla_inicio_disparo",  # 261 pulsos de rotura
    "tipo_falla_evento": "falla_estado_causa" # 1.661 horas de parada en taller
}, inplace=True)

# ------------------------------------------------------------------------------
# 4. GENERACIÓN DE SEÑALES DE CONTROL Y ALARMAS SCADA / PLC
# ------------------------------------------------------------------------------
print("[5/8] Generando registros de alarmas secuenciales de PLC/SCADA...")
alarmas = []
for _, row in df.iterrows():
    if row["estado_operativo"] == 0:
        alarmas.append("ALARM_MAQUINA_APAGADA")
    elif row["falla_inicio_disparo"] == 1:
        alarmas.append("TRIP_PARADA_EMERGENCIA")
    elif row["falla_proximas_48h"] == 1 and row["horas_hasta_falla"] <= 12:
        tipo = row["tipo_falla_inminente"]
        if tipo == "Fallo_Rodamiento":
            alarmas.append("ALARM_VIB_CRITICA")
        elif tipo == "Fallo_Motor_Termico":
            alarmas.append("ALARM_SOBRETEMPERATURA_MOTOR")
        elif tipo == "Fallo_Presion_Bomba":
            alarmas.append("ALARM_PRESION_ANORMAL")
        else:
            alarmas.append("ALARM_GENERAL_CRITICA")
    elif row["falla_proximas_48h"] == 1 and row["horas_hasta_falla"] <= 48:
        alarmas.append("WARN_ANOMALIA_TENDENCIA")
    else:
        # Falso positivo leve del 0.8% típico en plantas industriales
        if np.random.rand() < 0.008:
            alarmas.append("WARN_SOBRECARGA_LEVE")
        else:
            alarmas.append("SISTEMA_NORMAL")

df["codigo_alarma_plc"] = alarmas

# ------------------------------------------------------------------------------
# 5. CONSTRUCCIÓN DE TARGETS DE MACHINE LEARNING (SIN LEAKAGE)
# ------------------------------------------------------------------------------
print("[6/8] Estandarizando variables objetivo de supervisión...")
df["target_falla_48h"] = df["falla_proximas_48h"]
df["target_tipo_falla"] = df["tipo_falla_inminente"]
# Se sustituye la trampa de 999.0 por NaN para modelos de RUL continuo
df["target_rul_horas"] = df["horas_hasta_falla"].replace(999.0, np.nan)
df["target_estado_salud"] = df["estado_salud"]

# Limpieza de columnas intermedias redundantes
df.drop(columns=["falla_proximas_48h", "tipo_falla_inminente", "horas_hasta_falla", "estado_salud"], inplace=True)

# ------------------------------------------------------------------------------
# 6. INYECCIÓN CONTROLADA DE IMPERFECCIONES (GROUND TRUTH)
# ------------------------------------------------------------------------------
print("[7/8] Inyectando anomalías reales para validación del pipeline de limpieza...")
mask_operando = df["estado_operativo"] == 1

# A. Picos y Glitches de Vibración (280 eventos transitorios de 1 hora)
idx_spikes_vib = np.random.choice(df[mask_operando].index, size=280, replace=False)
df.loc[idx_spikes_vib, "vibracion_mms"] = np.random.uniform(25.0, 52.0, size=len(idx_spikes_vib))

# B. Transitorios de Voltaje (220 eventos de 1 hora)
idx_spikes_volt = np.random.choice(df[mask_operando].index, size=220, replace=False)
valores_volt = np.random.choice([np.random.uniform(340, 390), np.random.uniform(120, 150)], size=len(idx_spikes_volt))
df.loc[idx_spikes_volt, "voltaje_v"] = valores_volt

# C. Picos de Temperatura (150 eventos de 1 hora por termopar defectuoso)
idx_spikes_temp = np.random.choice(df[mask_operando].index, size=150, replace=False)
df.loc[idx_spikes_temp, "temperatura_c"] = np.random.uniform(130.0, 175.0, size=len(idx_spikes_temp))

# D. Deriva de Calibración Lenta (Sensor Drift)
# M-05: Sensor de temperatura acumula +0.035 °C por día
mask_m05 = df["id_maquina"] == "M-05"
dias_m05 = (df.loc[mask_m05, "fecha_hora"] - df.loc[mask_m05, "fecha_hora"].min()).dt.total_seconds() / (24 * 3600)
df.loc[mask_m05, "temperatura_c"] += dias_m05 * 0.035

# M-14: Sensor de presión decae -0.003 bar por día por ensuciamiento
mask_m14 = df["id_maquina"] == "M-14"
dias_m14 = (df.loc[mask_m14, "fecha_hora"] - df.loc[mask_m14, "fecha_hora"].min()).dt.total_seconds() / (24 * 3600)
df.loc[mask_m14, "presion_bar"] = np.maximum(0.1, df.loc[mask_m14, "presion_bar"] - (dias_m14 * 0.003))

# E. Sensores Congelados (Flatline / Varianza Cero)
# M-11: Temperatura fija en 62.45 °C durante 18 horas
mask_flat_m11 = (df["id_maquina"] == "M-11") & (df["fecha_hora"] >= "2026-02-15 00:00:00") & (df["fecha_hora"] <= "2026-02-15 18:00:00")
df.loc[mask_flat_m11, "temperatura_c"] = 62.45

# M-20: Vibración fija en 3.15 mm/s durante 16 horas
mask_flat_m20 = (df["id_maquina"] == "M-20") & (df["fecha_hora"] >= "2026-03-10 06:00:00") & (df["fecha_hora"] <= "2026-03-10 22:00:00")
df.loc[mask_flat_m20, "vibracion_mms"] = 3.15

# F. Valores Nulos Aleatorios (Pérdidas IoT - MCAR ~2.5% por sensor)
cols_sensores_nulos = ["temperatura_c", "vibracion_mms", "presion_bar", "voltaje_v", "carga_pct", "potencia_consumida_kw", "corriente_a"]
for col in cols_sensores_nulos:
    idx_nulos = np.random.choice(df.index, size=int(len(df) * 0.025), replace=False)
    df.loc[idx_nulos, col] = np.nan

# G. Desconexiones de Red en Ráfaga (Blackouts IoT - MNAR)
blackouts_red = [
    ("M-02", "2026-01-18 04:00:00", "2026-01-18 11:00:00"),
    ("M-06", "2026-02-04 12:00:00", "2026-02-04 18:00:00"),
    ("M-08", "2026-03-12 01:00:00", "2026-03-12 09:00:00"),
    ("M-17", "2026-01-29 14:00:00", "2026-01-29 22:00:00"),
    ("M-23", "2026-04-05 08:00:00", "2026-04-05 16:00:00")
]
cols_telemetria_blackout = ["carga_pct", "velocidad_rpm", "voltaje_v", "temperatura_c", "vibracion_mms", "presion_bar", "potencia_consumida_kw", "corriente_a"]
for maq, t_ini, t_fin in blackouts_red:
    mask_b = (df["id_maquina"] == maq) & (df["fecha_hora"] >= t_ini) & (df["fecha_hora"] <= t_fin)
    df.loc[mask_b, cols_telemetria_blackout] = np.nan

# ------------------------------------------------------------------------------
# 7. ORDENAMIENTO DE COLUMNAS Y FORMATEO DECIMAL
# ------------------------------------------------------------------------------
columnas_ordenadas = [
    # A. Metadatos del Activo (10)
    "fecha_hora", "id_maquina", "tipo_equipo", "modelo", "linea_produccion",
    "antiguedad_anos", "criticidad", "costo_parada_hora_usd", "potencia_nominal_kw", "marca",
    
    # B. Odómetros e Historial Operativo (4)
    "horas_operacion_totales", "ciclos_acumulados", "horas_desde_ultimo_mantenimiento", "conteo_fallas_previas",
    
    # C. Sensores Crudos en Tiempo Real (9)
    "estado_operativo", "carga_pct", "velocidad_rpm", "voltaje_v", "corriente_a",
    "potencia_consumida_kw", "temperatura_c", "vibracion_mms", "presion_bar",
    
    # D. Señales de Control (1)
    "codigo_alarma_plc",
    
    # E. Averías Industriales: Disparo vs Convalecencia (2)
    "falla_inicio_disparo", "falla_estado_causa",
    
    # F. Targets Predictivos para Machine Learning (4)
    "target_falla_48h", "target_tipo_falla", "target_rul_horas", "target_estado_salud"
]

df = df[columnas_ordenadas]

# Redondeo cosmético a 2 decimales para variables continuas
cols_redondeo = ["voltaje_v", "corriente_a", "potencia_consumida_kw", "temperatura_c", "vibracion_mms", "presion_bar", "carga_pct", "velocidad_rpm"]
for c in cols_redondeo:
    df[c] = df[c].round(2)

# ------------------------------------------------------------------------------
# 8. GUARDADO Y REPORTE DE AUDITORÍA
# ------------------------------------------------------------------------------
print("[8/8] Exportando dataset final y verificando métricas de integridad...")
df.to_csv(output_path, index=False)

print("\n" + "=" * 80)
print("REPORTE DE AUDITORÍA DEL DATASET GENERADO:")
print("=" * 80)
print(f"Dimensiones finales: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Archivo guardado en:\n  -> {output_path}")

print("\n--- Verificación de Nomenclatura de Averías ---")
print(f" • falla_inicio_disparo == 1: {(df['falla_inicio_disparo'] == 1).sum()} filas (Pulsos de rotura)")
print(f" • falla_estado_causa != 'Ninguna': {(df['falla_estado_causa'] != 'Ninguna').sum()} filas (Horas en taller)")
print(f" • MTTR Promedio: {((df['falla_estado_causa'] != 'Ninguna').sum() / (df['falla_inicio_disparo'] == 1).sum()):.2f} horas/falla")

print("\n--- Conteo de Valores Nulos Inyectados ---")
for col, nulos in df.isnull().sum()[df.isnull().sum() > 0].items():
    print(f" • {col:25s}: {nulos:,} nulos ({nulos/len(df)*100:.2f}%)")

print("\n--- Distribución del Target Alerta a 48h (MVP) ---")
print(df["target_falla_48h"].value_counts(dropna=False))

print("=" * 80)
print("PIPELINE EJECUTADO EXITOSAMENTE.")
print("=" * 80)